# Experiment 1.1 — Intrinsic Dataset Quality Evaluation

## Research Question
How can we assess the intrinsic quality of an automatically generated concept dataset?

This notebook uses a **VLM-as-judge** to measure how well each image actually depicts its concept. The judge is `Qwen2-VL-2B-Instruct`.

1. **Prompt:** The judge is asked whether `{concept}` is the *dominant* visible pattern covering most of the image *with no other competing pattern*, and is told to answer "yes" only if both conditions hold.

2. **Logit-based scoring:** Rather than letting the VLM generate a free-form yes/no answer, we run a single forward pass and read the next-token logits at the position the assistant would begin its reply. We softmax those logits over the candidate tokens for "Yes" / "No" (all casing variants combined via `logsumexp`) to obtain a continuous **P(yes | image, concept)** score per image. This is the standard approach in G-Eval, Prometheus, and the broader LLM-as-judge literature; it removes generation sycophancy and gives back a calibrated alignment score.

CLIP is still used as a frozen feature extractor for two embedding-based dataset-quality metrics (repetition rate, participation ratio) — these answer questions about the *embedding space* of the dataset (duplication, spread), not about whether each image matches the concept.

### Metrics computed per concept (Baseline vs. Automatic dataset):
| Metric | Description |
|--------|-------------|
| **VLM Alignment — Mean** | Mean P(yes) across images | 
| **VLM Alignment — Std** | Spread of P(yes) |
| **VLM Alignment — % below 0.25** | Fraction of clearly off-concept images |
| **VLM Precision @ 0.5** | % of images with P(yes) ≥ 0.5 |
| **VLM Precision @ 0.7** | % of images with P(yes) ≥ 0.7 (strict) |
| **Repetition Rate** | % of near-duplicate pairs (CLIP cosine sim > τ=0.95) |
| **Participation Ratio** | Effective rank of CLIP embedding covariance (diversity proxy) |

In [1]:
!pip install --quiet open_clip_torch pandas "transformers>=4.45" accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 21.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.4 MB/s eta 0:00:00


In [2]:
import os
import json
import shutil
import tempfile
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
import torch
import open_clip
from PIL import Image

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# CLIP (used here ONLY as a frozen feature extractor for repetition rate and participation ratio — no concept scoring)
model_name  = "ViT-B-32"
pretrained  = os.environ.get("CLIP_PRETRAINED", "openai")

hf_token = os.environ.get("HF_TOKEN", None)
if hf_token:
    os.environ.setdefault("HUGGING_FACE_HUB_TOKEN", hf_token)
elif pretrained != "openai":
    print(f"WARNING: pretrained='{pretrained}' may require an HF token. Falling back to 'openai'.")
    pretrained = "openai"

clip_model, _, clip_preprocess = open_clip.create_model_and_transforms(
    model_name, pretrained=pretrained
)
clip_model = clip_model.to(device).eval()
print(f"CLIP loaded (embeddings only): {model_name} / {pretrained}")

Using device: cuda


open_clip_model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/open_clip/factory.py:450: UserWarning: QuickGELU mismatch between final model config (quick_gelu=False) and pretrained tag 'openai' (quick_gelu=True).
  warnings.warn(


CLIP loaded (embeddings only): ViT-B-32 / openai


In [3]:
# VLM-as-Judge: load Qwen2-VL-2B-Instruct
# Other choices could be: Qwen/Qwen2.5-VL-3B-Instruct, Qwen/Qwen2-VL-7B-Instruct
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor

VLM_MODEL_ID = os.environ.get("VLM_MODEL_ID", "Qwen/Qwen2-VL-2B-Instruct")
vlm_dtype    = torch.float16 if device == "cuda" else torch.float32

print(f"Loading VLM judge: {VLM_MODEL_ID}  (first run downloads ~5 GB) ...")
vlm_model = Qwen2VLForConditionalGeneration.from_pretrained(
    VLM_MODEL_ID,
    torch_dtype=vlm_dtype,
    device_map="auto",
).eval()
vlm_processor = AutoProcessor.from_pretrained(VLM_MODEL_ID)
print("VLM judge loaded.")

# Discover token IDs for "Yes" and "No" (all casing variants)
# Used by the logit-based judge to read P(yes) from the model's first-token logits.
def _discover_yn_token_ids(processor):
    tok = processor.tokenizer
    yes_ids, no_ids = [], []
    for v in ["Yes", " Yes", "yes", " yes", "YES"]:
        ids = tok.encode(v, add_special_tokens=False)
        if len(ids) == 1:
            yes_ids.append(ids[0])
    for v in ["No", " No", "no", " no", "NO"]:
        ids = tok.encode(v, add_special_tokens=False)
        if len(ids) == 1:
            no_ids.append(ids[0])
    return sorted(set(yes_ids)), sorted(set(no_ids))

YES_TOKEN_IDS, NO_TOKEN_IDS = _discover_yn_token_ids(vlm_processor)
print(f"Yes token IDs: {YES_TOKEN_IDS}")
print(f"No token IDs:  {NO_TOKEN_IDS}")
assert YES_TOKEN_IDS and NO_TOKEN_IDS, "Failed to find single-token Yes/No variants — judge will not work."

Loading VLM judge: Qwen/Qwen2-VL-2B-Instruct  (first run downloads ~5 GB) ...


config.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

VLM judge loaded.
Yes token IDs: [7414, 9454, 9693, 9834, 14004]
No token IDs:  [902, 2152, 2308, 2753, 8996]


## Section 3 — Dataset Paths Configuration

In [4]:
ACG_NEW = "/kaggle/input/datasets/alessandrocogollo/acg4cav-base-extraction/dataset"

CROP_METHODS = ["bbox", "center_mask", "largest_bbox", "sliding_window"]
SPLITS       = ["single", "multi", "random"]

DTD2_ROOT = "/kaggle/input/datasets/jmexpert/describable-textures-dataset-dtd/dtd/images"
T2I_ROOT  = "/kaggle/input/datasets/alessandrocogollo/zeroshot-t2i-concepts/concepts"

MANUAL_BASELINE: Dict[str, str] = {
    "striped":   f"{DTD2_ROOT}/striped",
    "dotted":    f"{DTD2_ROOT}/dotted",
    "chequered": f"{DTD2_ROOT}/chequered",
    "wood":      f"{T2I_ROOT}/wood/wood",
    "water":     f"{T2I_ROOT}/water/water",
    "braided":   f"{DTD2_ROOT}/braided",
    "bubbly":    f"{T2I_ROOT}/bubbly/bubbly",
    "fibrous":   f"{DTD2_ROOT}/fibrous",
    "veined":    f"{DTD2_ROOT}/veined",
}

CONCEPTS_MAIN  = ["striped", "dotted", "chequered", "wood", "water", "braided", "bubbly", "fibrous", "veined"]
ALL_CONCEPTS   = CONCEPTS_MAIN # mappatura forzata

# Builder mapping
def build_mapping_auto_vanilla(concepts: List[str], crop_method: str, split: str) -> Dict[str, Dict[str, str]]:
    aug_root = f"{ACG_NEW}/augmented/augmented_{crop_method}"
    van_root = f"{ACG_NEW}/vanilla/vanilla_{crop_method}"
    return {
        c: {
            "augmented":    f"{aug_root}/{c}/{split}",
            "vanilla": f"{van_root}/{c}/{split}",
        }
        for c in concepts
    }

def build_mapping_manual(concepts: List[str]) -> Dict[str, Dict[str, str]]:
    return {c: {"manual": MANUAL_BASELINE[c]} for c in concepts}


THRESHOLD_REPETITION = 0.95
BATCH_SIZE           = 64
MAX_IMAGES_PER_DIR   = None
THRESHOLD_ALIGNMENT  = 0.25

print(f"Config OK — {len(CROP_METHODS)} crop methods × {len(SPLITS)} splits × {len(ALL_CONCEPTS)} concepts")
print(f"Total (crop, split) cells: {len(CROP_METHODS) * len(SPLITS)}")

Config OK — 4 crop methods × 3 splits × 9 concepts
Total (crop, split) cells: 12


## Section 4 — Core Utilities (CLIP embeddings + VLM judge)


In [5]:
# Image helpers
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}

def list_images(img_dir: str, max_n: int = MAX_IMAGES_PER_DIR) -> List[str]:
    """Return sorted list of image file paths inside img_dir, optionally capped at max_n."""
    paths = sorted(
        os.path.join(img_dir, f)
        for f in os.listdir(img_dir)
        if os.path.splitext(f.lower())[1] in IMAGE_EXTS
    )
    if max_n is not None:
        paths = paths[:max_n]
    return paths

# CLIP embeddings (for repetition rate + participation ratio only)

@torch.no_grad()
def embed_images(img_paths: List[str], batch_size: int = BATCH_SIZE) -> Tuple[np.ndarray, List[str]]:
    """
    Batch-embed images with CLIP.
    Skips unreadable files silently.
    Returns (N, D) L2-normalised float32 array and the valid path list.
    """
    embs, valid = [], []
    for i in range(0, len(img_paths), batch_size):
        batch_t, batch_p = [], []
        for p in img_paths[i : i + batch_size]:
            try:
                batch_t.append(clip_preprocess(Image.open(p).convert("RGB")))
                batch_p.append(p)
            except Exception:
                continue
        if not batch_t:
            continue
        feats = clip_model.encode_image(torch.stack(batch_t).to(device))
        feats = feats / feats.norm(dim=-1, keepdim=True)
        embs.append(feats.cpu().float().numpy())
        valid.extend(batch_p)
    if not embs:
        return np.zeros((0, clip_model.visual.output_dim), dtype=np.float32), []
    return np.vstack(embs).astype(np.float32), valid


def repetition_rate_from_embs(embs: np.ndarray, tau: float = THRESHOLD_REPETITION) -> dict:
    """Near-duplicate detection on pre-computed L2-normalised CLIP embeddings."""
    N = embs.shape[0]
    if N < 2:
        return {"N": N, "tau": tau, "repetition_rate": 0.0, "num_pairs": 0,
                "num_high_sim_pairs": 0, "max_offdiag_sim": None, "mean_offdiag_sim": None}
    S    = embs @ embs.T
    iu   = np.triu_indices(N, k=1)
    sims = S[iu]
    n_high = int((sims > tau).sum())
    return {
        "N": N,
        "tau": tau,
        "repetition_rate":   n_high / sims.size,
        "num_pairs":         int(sims.size),
        "num_high_sim_pairs": n_high,
        "max_offdiag_sim":   float(sims.max()),
        "mean_offdiag_sim":  float(sims.mean()),
    }


def participation_ratio_from_embs(embs: np.ndarray) -> dict:
    """
    Participation Ratio (PR) = (Σλ_i)² / Σ(λ_i²)  where λ_i are eigenvalues of the
    embedding covariance matrix.
    High PR → embeddings span many directions → diverse dataset.
    Low PR  → embeddings collapse to a few modes.
    """
    N = embs.shape[0]
    if N < 2:
        return {"N": N, "participation_ratio": float("nan")}
    Z_c     = embs - embs.mean(axis=0, keepdims=True)
    cov     = (Z_c.T @ Z_c) / (N - 1)
    eigvals = np.linalg.eigvalsh(cov)
    eigvals = eigvals[eigvals > 0]
    pr = float(eigvals.sum() ** 2 / (eigvals ** 2).sum())
    return {"N": N, "participation_ratio": pr}


# VLM-as-Judge (logit-based, strict prompt)

JUDGE_PROMPT_TMPL = (
    "Look at this image. Is '{concept}' the dominant visible pattern?\n"
    "Answer 'yes' only if the conditions hold; otherwise answer 'no'."
)

THRESHOLD_ALIGNMENT = 0.25  # P(yes) below this is considered "clearly off-concept"


@torch.inference_mode()
def vlm_judge_prob(img_path: str, concept: str) -> float:
    """
    Return P(yes | image, concept) from a single VLM forward pass.

    We read the next-token logits at the position the assistant begins its reply,
    select the logits at the candidate "Yes" and "No" token IDs (all casing
    variants), combine multi-variant logits with logsumexp, and softmax over
    the two combined scalars. This avoids sycophancy from generative decoding.

    Returns NaN for unreadable images.
    """
    try:
        img = Image.open(img_path).convert("RGB")
    except Exception:
        return float("nan")

    prompt_text = JUDGE_PROMPT_TMPL.format(concept=concept)
    messages = [{
        "role": "user",
        "content": [
            {"type": "image", "image": img},
            {"type": "text",  "text": prompt_text},
        ],
    }]
    text   = vlm_processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = vlm_processor(
        text=[text], images=[img], padding=True, return_tensors="pt",
    ).to(vlm_model.device)

    out         = vlm_model(**inputs)
    last_logits = out.logits[0, -1, :]                             # (vocab,)
    yes_lse     = torch.logsumexp(last_logits[YES_TOKEN_IDS], dim=0)
    no_lse      = torch.logsumexp(last_logits[NO_TOKEN_IDS],  dim=0)
    p_yes       = torch.softmax(torch.stack([yes_lse, no_lse]), dim=0)[0]
    return float(p_yes.item())


def vlm_judge_probs_batch(img_paths: List[str], concept: str) -> List[float]:
    """Return P(yes) per image. Sequential (one forward pass per image)."""
    try:
        from tqdm.auto import tqdm
    except ImportError:
        tqdm = lambda x, **k: x
    return [vlm_judge_prob(p, concept) for p in tqdm(img_paths, desc=f"VLM '{concept}'", leave=False)]


def compute_vlm_metrics(
    probs: List[float],
    threshold_alignment: float = THRESHOLD_ALIGNMENT,
    threshold_lenient: float = 0.5,
    threshold_strict: float  = 0.7,
) -> dict:
    """Aggregate per-image P(yes) into the alignment + precision metrics."""
    nan = float("nan")
    arr = np.array(probs, dtype=np.float32)
    arr = arr[~np.isnan(arr)]   # drop unreadable images
    n   = int(arr.size)
    if n == 0:
        return {
            "N": 0,
            "vlm_align_mean":      nan,
            "vlm_align_std":       nan,
            "vlm_align_pct_below": nan,
            "vlm_prec_05":         nan,
            "vlm_prec_07":         nan,
        }
    return {
        "N":                   n,
        "vlm_align_mean":      float(arr.mean()),
        "vlm_align_std":       float(arr.std(ddof=1)) if n > 1 else nan,
        "vlm_align_pct_below": float((arr < threshold_alignment).mean() * 100),
        "vlm_prec_05":         float((arr >= threshold_lenient).mean() * 100),
        "vlm_prec_07":         float((arr >= threshold_strict).mean()  * 100),
    }


## Section 5 — Main Experiment Loop

Results are **checkpointed** to `RESULTS_PATH` after each concept,
so a crashed or interrupted run can be resumed without re-embedding everything.


In [6]:
# Run all metrics for every concept
def experiments_run(MAPPINGS, RESULTS_PATH, tag, force=False):
    all_results = {}

    # Resume from checkpoint if available
    if os.path.exists(RESULTS_PATH):
        with open(RESULTS_PATH) as f:
            all_results = json.load(f)
        print(f"Resumed from checkpoint — {len(all_results)} concept(s) already done.")

    for concept, paths_dict in MAPPINGS.items():
        key = f"{tag}:{concept}"
    
        if not force and key in all_results:
            print(f"[SKIP] {concept}  (checkpoint)")
            continue
    
        print(f"\n{'='*60}\n  Concept: '{concept}'\n{'='*60}")
        concept_result = {}
    
        for split_name, path in paths_dict.items():
            if not path or not os.path.isdir(path):
                print(f"  [{split_name}] Path missing — skipping.")
                concept_result[split_name] = None
                continue

            paths  = list_images(path)
            n_imgs = len(paths)
            print(f"  [{split_name}] {n_imgs} images  |  {path}")

            print(f"  [{split_name}] Embedding images (CLIP, for diversity metrics) ...")
            embs, paths_valid = embed_images(paths)

            print(f"  [{split_name}] VLM-as-judge logit scoring ({len(paths_valid)} images) ...")
            judge_probs   = vlm_judge_probs_batch(paths_valid, concept)
            judge_metrics = compute_vlm_metrics(judge_probs)

            print(f"  [{split_name}] Repetition rate ...")
            rep = repetition_rate_from_embs(embs)

            print(f"  [{split_name}] Participation ratio (diversity) ...")
            div = participation_ratio_from_embs(embs)

            concept_result[split_name] = {
                "judge_probs":   judge_probs,
                "judge_metrics": judge_metrics,
                "rep":           rep,
                "div":           div,
            }
            print(f"  [{split_name}] Align μ={judge_metrics['vlm_align_mean']:.3f}  "
                  f"σ={judge_metrics['vlm_align_std']:.3f}  "
                  f"%<0.25: {judge_metrics['vlm_align_pct_below']:.1f}%  "
                  f"P@0.5: {judge_metrics['vlm_prec_05']:.1f}%  "
                  f"P@0.7: {judge_metrics['vlm_prec_07']:.1f}%  "
                  f"Rep: {rep['repetition_rate']*100:.2f}%  "
                  f"PR: {div['participation_ratio']:.2f}")

        all_results[key] = concept_result

        with tempfile.NamedTemporaryFile("w", delete=False, suffix=".json") as tmp:
            json.dump(all_results, tmp)
        shutil.move(tmp.name, RESULTS_PATH)

    print("\n✓ All concepts processed.")
    return all_results

In [7]:
manual_results = experiments_run(
    build_mapping_manual(ALL_CONCEPTS),
    "results_manual.json",
    "manual",
)

auto_vanilla_results: Dict[str, dict] = {}   # tag → results dict

for crop_method in CROP_METHODS:
    for split in SPLITS:
        tag = f"{crop_method}:{split}"
        print(f"\n{'#'*70}\n#  {tag}\n{'#'*70}")
        auto_vanilla_results[tag] = experiments_run(
            build_mapping_auto_vanilla(CONCEPTS_MAIN, crop_method, split),
            f"results_{crop_method}_{split}.json",
            tag,
        )


  Concept: 'striped'
  [manual] 120 images  |  /kaggle/input/datasets/jmexpert/describable-textures-dataset-dtd/dtd/images/striped
  [manual] Embedding images (CLIP, for diversity metrics) ...
  [manual] VLM-as-judge logit scoring (120 images) ...


VLM 'striped':   0%|          | 0/120 [00:00<?, ?it/s]

  [manual] Repetition rate ...
  [manual] Participation ratio (diversity) ...
  [manual] Align μ=0.872  σ=0.077  %<0.25: 0.0%  P@0.5: 100.0%  P@0.7: 95.0%  Rep: 0.57%  PR: 15.42

  Concept: 'dotted'
  [manual] 120 images  |  /kaggle/input/datasets/jmexpert/describable-textures-dataset-dtd/dtd/images/dotted
  [manual] Embedding images (CLIP, for diversity metrics) ...
  [manual] VLM-as-judge logit scoring (120 images) ...


VLM 'dotted':   0%|          | 0/120 [00:00<?, ?it/s]

  [manual] Repetition rate ...
  [manual] Participation ratio (diversity) ...
  [manual] Align μ=0.803  σ=0.095  %<0.25: 0.0%  P@0.5: 98.3%  P@0.7: 89.2%  Rep: 0.24%  PR: 26.23

  Concept: 'chequered'
  [manual] 120 images  |  /kaggle/input/datasets/jmexpert/describable-textures-dataset-dtd/dtd/images/chequered
  [manual] Embedding images (CLIP, for diversity metrics) ...
  [manual] VLM-as-judge logit scoring (120 images) ...


...

  [vanilla] Repetition rate ...
  [vanilla] Participation ratio (diversity) ...
  [vanilla] Align μ=0.390  σ=0.165  %<0.25: 25.0%  P@0.5: 26.0%  P@0.7: 4.0%  Rep: 0.00%  PR: 32.40

  Concept: 'veined'
  [augmented] 100 images  |  /kaggle/input/datasets/alessandrocogollo/acg4cav-base-extraction/dataset/augmented/augmented_sliding_window/veined/random
  [augmented] Embedding images (CLIP, for diversity metrics) ...
  [augmented] VLM-as-judge logit scoring (100 images) ...


VLM 'veined':   0%|          | 0/100 [00:00<?, ?it/s]

  [augmented] Repetition rate ...
  [augmented] Participation ratio (diversity) ...
  [augmented] Align μ=0.497  σ=0.147  %<0.25: 5.0%  P@0.5: 48.0%  P@0.7: 7.0%  Rep: 0.08%  PR: 30.36
  [vanilla] 100 images  |  /kaggle/input/datasets/alessandrocogollo/acg4cav-base-extraction/dataset/vanilla/vanilla_sliding_window/veined/random
  [vanilla] Embedding images (CLIP, for diversity metrics) ...
  [vanilla] VLM-as-judge logit scoring (100 images) ...


VLM 'veined':   0%|          | 0/100 [00:00<?, ?it/s]

  [vanilla] Repetition rate ...
  [vanilla] Participation ratio (diversity) ...
  [vanilla] Align μ=0.421  σ=0.171  %<0.25: 15.0%  P@0.5: 34.0%  P@0.7: 5.0%  Rep: 0.00%  PR: 32.52

✓ All concepts processed.


## Section 6 — Results
An aggregate table (and then CSV) with the results

In [8]:
import pandas as pd
import numpy as np

try:
    from IPython.display import display as ipy_display
    _display = ipy_display
except ImportError:
    _display = print

THRESHOLD_ALIGNMENT = 0.25
METRIC_INFO = [
    ("VLM Alignment — Mean",                          "vlm_align_mean",      "↑"),
    ("VLM Alignment — Std",                           "vlm_align_std",       "↓"),
    (f"VLM Alignment — % below {THRESHOLD_ALIGNMENT}", "vlm_align_pct_below", "↓"),
    ("VLM Precision @ 0.5 (%)",                       "vlm_prec_05",         "↑"),
    ("VLM Precision @ 0.7 (%)",                       "vlm_prec_07",         "↑"),
    ("Repetition Rate (%)",                           "rep_rate",            "↓"),
    ("Num Near-Duplicate Pairs",                      "n_dup_pairs",         "↓"),
    ("Participation Ratio",                           "participation_ratio", "↑"),
    ("Dataset Size (N)",                              "N",                   "·"),
]

METRIC_INFO_MAP = {label: direction for label, _, direction in METRIC_INFO}

def _split_metrics(split_data) -> dict:
    nan = float("nan")
    if split_data is None or not isinstance(split_data, dict):
        return {k: nan for k in [
            "vlm_align_mean", "vlm_align_std", "vlm_align_pct_below",
            "vlm_prec_05", "vlm_prec_07",
            "rep_rate", "n_dup_pairs", "participation_ratio", "N"
        ]}
    judge = split_data.get("judge_metrics", {}) or {}
    rep   = split_data.get("rep", {"repetition_rate": nan, "num_high_sim_pairs": nan})
    div   = split_data.get("div", {"participation_ratio": nan, "N": nan})
    
    return {
        "vlm_align_mean":      judge.get("vlm_align_mean",      nan),
        "vlm_align_std":       judge.get("vlm_align_std",       nan),
        "vlm_align_pct_below": judge.get("vlm_align_pct_below", nan),
        "vlm_prec_05":         judge.get("vlm_prec_05",         nan),
        "vlm_prec_07":         judge.get("vlm_prec_07",         nan),
        "rep_rate":            rep["repetition_rate"] * 100 if not np.isnan(rep["repetition_rate"]) else nan,
        "n_dup_pairs":         rep["num_high_sim_pairs"],
        "participation_ratio": div["participation_ratio"],
        "N":                   div["N"],
    }

def _highlight_best_overall(df):
    def style_row(row):
        direction = METRIC_INFO_MAP.get(row.name, "·")
        styles = [""] * len(row)
        
        if direction in ["—", "·"]: 
            return styles
            
        cols_to_compare = [c for c in row.index if c != "Manual Baseline"]
        
        best_val = None
        for c in cols_to_compare:
            try:
                val = float(row[c])
                if not np.isnan(val):
                    if best_val is None:
                        best_val = val
                    else:
                        if direction == "↑":
                            best_val = max(best_val, val)
                        elif direction == "↓":
                            best_val = min(best_val, val)
            except:
                pass
                
        for i, c in enumerate(row.index):
            if c in cols_to_compare:
                try:
                    val = float(row[c])
                    if not np.isnan(val) and best_val is not None and np.isclose(val, best_val, atol=1e-5):
                        styles[i] = "background-color: #d6f5d6; font-weight: bold; color: #004d00;"
                except:
                    pass
        return styles
        
    return df.style.apply(style_row, axis=1)

all_concepts = set()
for strategy_results in auto_vanilla_results.values():
    for key in strategy_results.keys():
        all_concepts.add(key.split(":")[-1])
all_concepts = sorted(list(all_concepts))

print("  COMPREHENSIVE CONCEPT ANALYSIS (All Strategies vs Manual Baseline)")

for concept in all_concepts:
    cols_data = {}
    
    manual_cell = manual_results.get(f"manual:{concept}", {})
    cols_data["Manual Baseline"] = _split_metrics(manual_cell.get("manual"))
    
    for strategy_tag, strategy_results in auto_vanilla_results.items():
        cell_result = next((v for k, v in strategy_results.items() if k.endswith(f":{concept}") or k == concept), {})
        
        short_tag = strategy_tag.replace("augmented_", "").replace("vanilla_", "")
        
        cols_data[f"Van ({short_tag})"] = _split_metrics(cell_result.get("vanilla"))
        cols_data[f"Aug ({short_tag})"] = _split_metrics(cell_result.get("augmented"))

    df = pd.DataFrame(
        {col_name: [cols_data[col_name][key] for _, key, _ in METRIC_INFO] 
         for col_name in cols_data.keys()},
        index=[name for name, _, _ in METRIC_INFO]
    )
    df.index.name = "Metric"
    
    print(f"\n\n CONCEPT: '{concept.upper()}'")
    
    styled_df = _highlight_best_overall(df).format(precision=3, na_rep="NaN")
    _display(styled_df)

  COMPREHENSIVE CONCEPT ANALYSIS (All Strategies vs Manual Baseline)


 CONCEPT: 'BRAIDED'


,Manual Baseline,Van (bbox:single),Aug (bbox:single),Van (bbox:multi),Aug (bbox:multi),Van (bbox:random),Aug (bbox:random),Van (center_mask:single),Aug (center_mask:single),Van (center_mask:multi),Aug (center_mask:multi),Van (center_mask:random),Aug (center_mask:random),Van (largest_bbox:single),Aug (largest_bbox:single),Van (largest_bbox:multi),Aug (largest_bbox:multi),Van (largest_bbox:random),Aug (largest_bbox:random),Van (sliding_window:single),Aug (sliding_window:single),Van (sliding_window:multi),Aug (sliding_window:multi),Van (sliding_window:random),Aug (sliding_window:random)
Metric,,,,,,,,,,,,,,,,,,,,,,,,,
VLM Alignment — Mean,0.888,0.410,0.484,0.655,0.597,0.236,0.313,0.436,0.496,0.659,0.656,0.283,0.324,0.424,0.491,0.661,0.649,0.294,0.350,0.427,0.477,0.633,0.571,0.243,0.336
VLM Alignment — Std,0.073,0.246,0.235,0.211,0.201,0.159,0.140,0.259,0.233,0.202,0.161,0.174,0.137,0.241,0.223,0.196,0.163,0.174,0.158,0.265,0.209,0.221,0.198,0.170,0.157
VLM Alignment — % below 0.25,0.000,37.000,21.000,7.000,8.000,63.000,40.000,29.000,17.000,6.000,2.000,50.000,29.000,32.000,19.000,5.000,2.000,45.000,31.000,35.000,15.000,12.000,5.000,62.000,35.000
VLM Precision @ 0.5 (%),99.167,39.000,54.000,83.000,71.000,7.000,11.000,42.000,48.000,80.000,86.000,12.000,10.000,39.000,46.000,83.000,81.000,15.000,18.000,42.000,44.000,79.000,64.000,11.000,16.000
VLM Precision @ 0.7 (%),97.500,20.000,27.000,56.000,41.000,2.000,0.000,27.000,27.000,60.000,47.000,0.000,0.000,22.000,30.000,55.000,46.000,1.000,3.000,25.000,20.000,56.000,37.000,3.000,3.000
Repetition Rate (%),0.014,0.020,0.000,0.000,0.020,0.020,0.020,0.000,0.020,0.000,0.000,0.000,0.081,0.000,0.040,0.000,0.040,0.000,0.061,0.000,0.020,0.000,0.081,0.000,0.061
Num Near-Duplicate Pairs,1.000,1.000,0.000,0.000,1.000,1.000,1.000,0.000,1.000,0.000,0.000,0.000,4.000,0.000,2.000,0.000,2.000,0.000,3.000,0.000,1.000,0.000,4.000,0.000,3.000
Participation Ratio,23.392,29.663,30.477,24.047,28.168,33.170,29.977,31.599,30.414,26.398,31.429,33.101,29.954,29.601,29.402,24.862,29.003,30.815,29.649,32.453,27.786,27.624,24.768,34.122,34.058
Dataset Size (N),120.000,100.000,100.000,100.000,100.000,100.000,100.000,100.000,100.000,100.000,100.000,100.000,100.000,100.000,100.000,100.000,100.000,100.000,100.000,100.000,100.000,100.000,100.000,100.000,100.000




 CONCEPT: 'BUBBLY'


,Manual Baseline,Van (bbox:single),Aug (bbox:single),Van (bbox:multi),Aug (bbox:multi),Van (bbox:random),Aug (bbox:random),Van (center_mask:single),Aug (center_mask:single),Van (center_mask:multi),Aug (center_mask:multi),Van (center_mask:random),Aug (center_mask:random),Van (largest_bbox:single),Aug (largest_bbox:single),Van (largest_bbox:multi),Aug (largest_bbox:multi),Van (largest_bbox:random),Aug (largest_bbox:random),Van (sliding_window:single),Aug (sliding_window:single),Van (sliding_window:multi),Aug (sliding_window:multi),Van (sliding_window:random),Aug (sliding_window:random)
Metric,,,,,,,,,,,,,,,,,,,,,,,,,
VLM Alignment — Mean,0.890,0.629,0.597,0.616,0.534,0.593,0.467,0.671,0.645,0.639,0.559,0.599,0.471,0.658,0.606,0.622,0.534,0.585,0.440,0.601,0.539,0.590,0.477,0.591,0.447
VLM Alignment — Std,0.048,0.113,0.115,0.165,0.163,0.210,0.145,0.119,0.127,0.164,0.169,0.208,0.164,0.097,0.125,0.169,0.172,0.210,0.173,0.137,0.141,0.164,0.158,0.212,0.137
VLM Alignment — % below 0.25,0.000,0.000,0.000,2.000,3.000,5.000,2.000,0.000,0.000,2.020,2.000,3.000,2.000,0.000,0.000,2.000,3.000,6.000,12.000,1.000,1.000,2.000,5.000,5.000,1.000
VLM Precision @ 0.5 (%),100.000,84.000,77.000,76.000,55.000,65.000,32.000,89.000,83.000,77.778,64.000,65.000,37.000,93.000,78.000,78.000,53.000,64.000,33.000,77.000,60.000,68.000,41.000,65.000,30.000
VLM Precision @ 0.7 (%),98.333,26.000,21.000,37.000,17.000,38.000,7.000,47.000,38.000,45.455,25.000,40.000,15.000,36.000,26.000,39.000,22.000,38.000,11.000,21.000,16.000,28.000,8.000,39.000,6.000
Repetition Rate (%),0.112,0.000,0.061,0.020,0.040,0.000,0.061,0.000,0.040,0.000,0.040,0.020,0.222,0.000,0.141,0.040,0.040,0.020,0.242,0.000,0.222,0.020,0.222,0.020,0.081
Num Near-Duplicate Pairs,8.000,0.000,3.000,1.000,2.000,0.000,3.000,0.000,2.000,0.000,2.000,1.000,11.000,0.000,7.000,2.000,2.000,1.000,12.000,0.000,11.000,1.000,11.000,1.000,4.000
Participation Ratio,19.685,21.941,20.114,14.844,23.897,12.876,26.507,26.289,26.229,17.663,28.688,12.296,25.505,24.435,24.828,15.495,26.918,12.035,25.335,22.333,24.934,18.321,25.239,11.511,24.166
Dataset Size (N),120.000,100.000,100.000,100.000,100.000,100.000,100.000,100.000,100.000,99.000,100.000,100.000,100.000,100.000,100.000,100.000,100.000,100.000,100.000,100.000,100.000,100.000,100.000,100.000,100.000




 CONCEPT: 'CHEQUERED'


...

,Manual Baseline,Van (bbox:single),Aug (bbox:single),Van (bbox:multi),Aug (bbox:multi),Van (bbox:random),Aug (bbox:random),Van (center_mask:single),Aug (center_mask:single),Van (center_mask:multi),Aug (center_mask:multi),Van (center_mask:random),Aug (center_mask:random),Van (largest_bbox:single),Aug (largest_bbox:single),Van (largest_bbox:multi),Aug (largest_bbox:multi),Van (largest_bbox:random),Aug (largest_bbox:random),Van (sliding_window:single),Aug (sliding_window:single),Van (sliding_window:multi),Aug (sliding_window:multi),Van (sliding_window:random),Aug (sliding_window:random)
Metric,,,,,,,,,,,,,,,,,,,,,,,,,
VLM Alignment — Mean,0.862,0.481,0.574,0.736,0.747,0.390,0.467,0.460,0.550,0.727,0.764,0.398,0.464,0.470,0.580,0.716,0.754,0.388,0.464,0.471,0.553,0.723,0.726,0.421,0.497
VLM Alignment — Std,0.086,0.202,0.183,0.204,0.173,0.171,0.161,0.208,0.182,0.223,0.171,0.170,0.135,0.194,0.162,0.224,0.179,0.181,0.155,0.216,0.175,0.208,0.180,0.171,0.147
VLM Alignment — % below 0.25,0.000,12.000,3.000,3.000,0.000,24.000,7.000,18.000,3.000,5.000,1.000,22.000,3.000,13.000,1.000,5.000,0.000,25.000,6.000,14.000,4.000,5.000,2.000,15.000,5.000
VLM Precision @ 0.5 (%),99.167,42.000,61.000,86.000,88.000,27.000,39.000,43.000,56.000,80.000,90.000,26.000,40.000,41.000,65.000,80.000,83.000,27.000,37.000,36.000,61.000,82.000,86.000,34.000,48.000
VLM Precision @ 0.7 (%),95.833,18.000,33.000,70.000,69.000,6.000,10.000,16.000,26.000,73.000,74.000,2.000,4.000,14.000,24.000,66.000,70.000,3.000,8.000,22.000,25.000,73.000,74.000,5.000,7.000
Repetition Rate (%),0.098,0.020,0.020,0.121,0.020,0.000,0.040,0.000,0.040,0.000,0.081,0.000,0.000,0.040,0.061,0.121,0.000,0.000,0.141,0.000,0.222,0.020,0.101,0.000,0.081
Num Near-Duplicate Pairs,7.000,1.000,1.000,6.000,1.000,0.000,2.000,0.000,2.000,0.000,4.000,0.000,0.000,2.000,3.000,6.000,0.000,0.000,7.000,0.000,11.000,1.000,5.000,0.000,4.000
Participation Ratio,21.453,17.614,24.514,11.873,16.583,32.483,29.410,20.054,24.375,15.407,19.647,33.578,30.972,17.042,21.067,11.918,14.957,32.041,30.463,24.137,27.094,20.304,18.203,32.520,30.357
Dataset Size (N),120.000,100.000,100.000,100.000,100.000,100.000,100.000,100.000,100.000,100.000,100.000,100.000,100.000,100.000,100.000,100.000,100.000,100.000,100.000,100.000,100.000,100.000,100.000,100.000,100.000




 CONCEPT: 'WATER'


,Manual Baseline,Van (bbox:single),Aug (bbox:single),Van (bbox:multi),Aug (bbox:multi),Van (bbox:random),Aug (bbox:random),Van (center_mask:single),Aug (center_mask:single),Van (center_mask:multi),Aug (center_mask:multi),Van (center_mask:random),Aug (center_mask:random),Van (largest_bbox:single),Aug (largest_bbox:single),Van (largest_bbox:multi),Aug (largest_bbox:multi),Van (largest_bbox:random),Aug (largest_bbox:random),Van (sliding_window:single),Aug (sliding_window:single),Van (sliding_window:multi),Aug (sliding_window:multi),Van (sliding_window:random),Aug (sliding_window:random)
Metric,,,,,,,,,,,,,,,,,,,,,,,,,
VLM Alignment — Mean,0.875,0.772,0.659,0.758,0.650,0.279,0.360,0.804,0.668,0.773,0.673,0.269,0.373,0.772,0.640,0.755,0.677,0.270,0.353,0.781,0.584,0.755,0.577,0.248,0.360
VLM Alignment — Std,0.056,0.092,0.153,0.095,0.160,0.169,0.158,0.066,0.153,0.107,0.170,0.163,0.149,0.093,0.169,0.096,0.151,0.182,0.148,0.141,0.176,0.156,0.193,0.179,0.156
VLM Alignment — % below 0.25,0.000,1.000,2.000,0.000,0.000,53.000,23.000,0.000,0.000,0.000,0.000,55.000,18.000,1.000,1.000,0.000,0.000,61.000,27.000,2.000,1.000,1.000,3.000,67.000,26.000
VLM Precision @ 0.5 (%),100.000,99.000,82.000,98.000,81.000,8.000,17.000,100.000,81.000,97.000,81.000,8.000,16.000,99.000,79.000,98.000,83.000,10.000,14.000,94.000,66.000,91.000,64.000,8.000,19.000
VLM Precision @ 0.7 (%),98.000,87.000,45.000,77.000,47.000,4.000,5.000,91.000,48.000,81.000,54.000,4.000,4.000,85.000,49.000,75.000,49.000,5.000,4.000,82.000,33.000,73.000,34.000,6.000,5.000
Repetition Rate (%),0.162,0.000,0.020,0.000,0.020,0.000,0.061,0.000,0.020,0.000,0.020,0.040,0.222,0.000,0.020,0.000,0.020,0.000,0.061,0.020,0.202,0.040,0.424,0.020,0.040
Num Near-Duplicate Pairs,8.000,0.000,1.000,0.000,1.000,0.000,3.000,0.000,1.000,0.000,1.000,2.000,11.000,0.000,1.000,0.000,1.000,0.000,3.000,1.000,10.000,2.000,21.000,1.000,2.000
Participation Ratio,19.733,25.700,31.139,26.865,27.503,31.485,30.639,24.534,27.033,27.470,28.537,31.004,29.596,25.732,28.877,26.732,28.566,32.835,30.553,23.519,24.117,22.411,23.813,32.823,32.475
Dataset Size (N),100.000,100.000,100.000,100.000,100.000,100.000,100.000,100.000,100.000,100.000,100.000,100.000,100.000,100.000,100.000,100.000,100.000,100.000,100.000,100.000,100.000,100.000,100.000,100.000,100.000




 CONCEPT: 'WOOD'


,Manual Baseline,Van (bbox:single),Aug (bbox:single),Van (bbox:multi),Aug (bbox:multi),Van (bbox:random),Aug (bbox:random),Van (center_mask:single),Aug (center_mask:single),Van (center_mask:multi),Aug (center_mask:multi),Van (center_mask:random),Aug (center_mask:random),Van (largest_bbox:single),Aug (largest_bbox:single),Van (largest_bbox:multi),Aug (largest_bbox:multi),Van (largest_bbox:random),Aug (largest_bbox:random),Van (sliding_window:single),Aug (sliding_window:single),Van (sliding_window:multi),Aug (sliding_window:multi),Van (sliding_window:random),Aug (sliding_window:random)
Metric,,,,,,,,,,,,,,,,,,,,,,,,,
VLM Alignment — Mean,0.855,0.731,0.653,0.667,0.548,0.303,0.325,0.730,0.662,0.661,0.557,0.272,0.317,0.738,0.680,0.663,0.554,0.266,0.317,0.673,0.574,0.680,0.509,0.279,0.297
VLM Alignment — Std,0.092,0.121,0.158,0.189,0.205,0.212,0.168,0.119,0.150,0.190,0.192,0.180,0.174,0.109,0.146,0.192,0.197,0.190,0.165,0.187,0.178,0.164,0.198,0.199,0.146
VLM Alignment — % below 0.25,0.000,1.000,2.000,8.000,11.000,53.000,38.000,1.000,0.000,7.000,5.000,57.000,42.000,0.000,0.000,8.000,11.000,57.000,42.000,6.000,3.000,1.000,9.000,54.000,39.000
VLM Precision @ 0.5 (%),98.000,93.000,77.000,87.000,60.000,19.000,15.000,96.000,82.000,85.000,60.000,13.000,16.000,94.000,86.000,87.000,64.000,12.000,15.000,83.000,64.000,83.000,48.000,18.000,5.000
VLM Precision @ 0.7 (%),98.000,73.000,44.000,57.000,33.000,5.000,3.000,68.000,49.000,56.000,29.000,4.000,3.000,72.000,55.000,58.000,30.000,4.000,4.000,62.000,32.000,60.000,21.000,5.000,2.000
Repetition Rate (%),0.000,0.000,0.081,0.000,0.040,0.000,0.061,0.000,0.020,0.000,0.061,0.000,0.020,0.000,0.061,0.000,0.020,0.000,0.101,0.000,0.081,0.040,0.081,0.000,0.081
Num Near-Duplicate Pairs,0.000,0.000,4.000,0.000,2.000,0.000,3.000,0.000,1.000,0.000,3.000,0.000,1.000,0.000,3.000,0.000,1.000,0.000,5.000,0.000,4.000,2.000,4.000,0.000,4.000
Participation Ratio,28.037,28.808,25.889,20.873,27.978,32.592,27.918,28.350,28.387,22.495,29.135,32.674,30.325,32.509,29.495,21.428,27.027,31.380,32.462,27.950,26.882,23.150,27.275,32.641,30.196
Dataset Size (N),100.000,100.000,100.000,100.000,100.000,100.000,100.000,100.000,100.000,100.000,100.000,100.000,100.000,100.000,100.000,100.000,100.000,100.000,100.000,100.000,100.000,100.000,100.000,100.000,100.000
